# Task 11 — LMR Period and Event Fingerprints

Sanity-check LMR v2.1 against known historical climate anomalies (MCA, LIA) and major volcanic
events from eVolv2k, then establish a baseline-window convention for Band T anomaly reporting.

**Building on Task 10**: uses the 34-cell geographic sample (`10_sample_locations.csv`),
same file paths and loading pattern.

**Caveat (F10.1)**: LMR reconstruction fidelity is reduced before ~700 CE (funnel/regression-to-prior
effect). The late-antique period (500–700 CE) is included for completeness but flagged throughout.

**Outputs**: `11_period_anomalies.{csv,png}`, `11_volcanic_response_cells.{csv,png}`,
`11_volcanic_response_nhmt.png`, `11_baseline_convention.csv`

In [2]:
# Cell 2 — Imports and config
import warnings
import numpy as np
import pandas as pd
import netCDF4 as nc
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import time as _time
from pathlib import Path
import sys
sys.path.insert(0, "/Users/karlg/Documents/Repos/_cedop")

LMR_DIR  = Path("/Users/karlg/Documents/Repos/_cedop/data/lmr_v2.1")
OUT_DIR  = Path("/Users/karlg/Documents/Repos/_cedop/output/edop/explore")
EVOL_CSV = Path("/Users/karlg/Documents/Repos/_cedop/data/volcano/evolv2k_v4.csv")

MEAN_FILES = {
    "pdsi": LMR_DIR / "pdsi_MCruns_ensemble_mean_LMRv2.1.nc",
    "air":  LMR_DIR / "air_MCruns_ensemble_mean_LMRv2.1.nc",
}
NHMT_FILE = LMR_DIR / "nhmt_MCruns_ensemble_full_LMRv2.1.nc"
GMT_FILE  = LMR_DIR / "gmt_MCruns_ensemble_full_LMRv2.1.nc"

# Time axis (shared across variables)
ds = nc.Dataset(MEAN_FILES["pdsi"])
time_var = ds.variables["time"]
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    times = nc.num2date(time_var[:], time_var.units, has_year_zero=True)
years = np.array([t.year for t in times])   # 0–1998 CE
lats  = ds.variables["lat"][:]
lons  = ds.variables["lon"][:]
ds.close()

# Sample locations from Task 10 (34 cells, L8-anchored)
sample_df = pd.read_csv(OUT_DIR / "10_sample_locations.csv")

# eVolv2k catalog
evolv = pd.read_csv(EVOL_CSV)
evolv["vssi_tg"] = pd.to_numeric(evolv["vssi_tg"], errors="coerce")
evolv = evolv.dropna(subset=["vssi_tg"]).copy()

BAND_COLORS = {
    "Arctic": "#1f77b4", "Temperate NH": "#ff7f0e", "Subtropical NH": "#2ca02c",
    "Tropical N": "#d62728", "Tropical S": "#9467bd",
    "Subtropical SH": "#8c564b", "Temperate SH": "#e377c2"
}

print(f"Time axis: {years[0]}–{years[-1]} CE  ({len(years)} steps)")
print(f"Sample locations: {len(sample_df)} cells")
print(f"eVolv2k events: {len(evolv)} total")
print(sample_df.groupby("band").size().to_string())

Time axis: 0–1998 CE  (2001 steps)
Sample locations: 34 cells
eVolv2k events: 256 total
band
Arctic             4
Subtropical NH     5
Subtropical SH     3
Temperate NH      10
Temperate SH       4
Tropical N         4
Tropical S         4


In [3]:
# Cell 3 — Load grand-mean time series for 34 sample cells (pdsi + air)
# Reuses Task 10 extract_cell_series pattern.
# prate excluded: not needed for period/volcanic analysis.

def extract_cell_mean(nc_path, var_name, li, lj):
    """Grand mean (across 20 MCruns) for one grid cell. Returns array of shape (2001,)."""
    ds = nc.Dataset(nc_path)
    data = np.ma.filled(ds.variables[var_name][:, :, li, lj], np.nan).astype(np.float32)
    ds.close()
    return np.nanmean(data, axis=1)

print("Loading sample time series (pdsi + air)...")
t0 = _time.time()
series = {var: [] for var in ["pdsi", "air"]}

for var in ["pdsi", "air"]:
    print(f"  {var}...", end="", flush=True)
    for _, row in sample_df.iterrows():
        ts = extract_cell_mean(MEAN_FILES[var], var, int(row.lat_idx), int(row.lon_idx))
        series[var].append(ts)
    print(" done")

print(f"Done in {_time.time()-t0:.1f}s  ({len(series['air'])} cells x 2 variables)")

 done
Done in 16.5s  (34 cells x 2 variables)


In [4]:
# Cell 4 — Period anomaly extraction
# Anomaly = mean(ts, period) - mean(ts, reference).
# LMR values are already anomalies from the model prior; subtracting the reference
# mean cancels the prior offset and gives a clean period-vs-reference comparison.
# Primary reference: full record 0–1998 CE.  Alternatives compared in Cell 10.

PERIODS = {
    "Late Antique (500–700 CE)": (500, 700),    # in funnel zone — treat with caution
    "MCA (950–1250 CE)":         (950, 1250),
    "LIA (1300–1850 CE)":        (1300, 1850),
}
REF_MASK_FULL = np.ones(len(years), dtype=bool)  # full record 0–1998 CE

rows = []
for var in ["pdsi", "air"]:
    for i, cell in sample_df.iterrows():
        ts = series[var][i]
        ref_mean = np.nanmean(ts[REF_MASK_FULL])
        for period_name, (yr_lo, yr_hi) in PERIODS.items():
            mask = (years >= yr_lo) & (years <= yr_hi)
            period_mean = np.nanmean(ts[mask])
            rows.append({
                "variable":       var,
                "period":         period_name,
                "band":           cell["band"],
                "lat":            cell["lat"],
                "lon":            cell["lon"],
                "anomaly_vs_full": period_mean - ref_mean,
                "funnel_caution": "YES" if yr_hi <= 700 else "",
            })

anom_df = pd.DataFrame(rows)

print("=== Median period anomaly vs full-record mean ===")
print(f"{'Period':<30} {'Var':<6} {'Median anomaly':>16}  {'% cells positive':>16}")
print("-" * 74)
for period in PERIODS:
    for var in ["air", "pdsi"]:
        sub = anom_df[(anom_df.period == period) & (anom_df.variable == var)]
        med = sub["anomaly_vs_full"].median()
        pct_pos = 100 * (sub["anomaly_vs_full"] > 0).mean()
        flag = "  ⚠ funnel" if "500" in period else ""
        print(f"{period:<30} {var:<6} {med:>16.4f}  {pct_pos:>15.1f}%{flag}")
    print()

=== Median period anomaly vs full-record mean ===
Period                         Var      Median anomaly  % cells positive
--------------------------------------------------------------------------
Late Antique (500–700 CE)      air              0.0159             73.5%  ⚠ funnel
Late Antique (500–700 CE)      pdsi            -0.0003             38.2%  ⚠ funnel

MCA (950–1250 CE)              air             -0.0408              8.8%
MCA (950–1250 CE)              pdsi             0.0266             61.8%

LIA (1300–1850 CE)             air             -0.0410             11.8%
LIA (1300–1850 CE)             pdsi            -0.0087             35.3%



In [5]:
# Cell 5 — Period anomaly plots: distribution by latitude band
BANDS_ORDER = ["Arctic", "Temperate NH", "Subtropical NH", "Tropical N",
               "Tropical S", "Subtropical SH", "Temperate SH"]
BAND_SHORT   = [b[:8] for b in BANDS_ORDER]
UNIT_LABEL   = {"air": "Temp anomaly (K)", "pdsi": "PDSI anomaly"}
PERIOD_LIST  = list(PERIODS.keys())

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharey="row")

for col, period in enumerate(PERIOD_LIST):
    for row, var in enumerate(["air", "pdsi"]):
        ax = axes[row][col]
        sub = anom_df[(anom_df.period == period) & (anom_df.variable == var)]
        data_by_band = [sub[sub.band == b]["anomaly_vs_full"].values for b in BANDS_ORDER]
        bp = ax.boxplot(data_by_band, labels=BAND_SHORT, patch_artist=True,
                        medianprops=dict(color="black", lw=1.5))
        for patch, color in zip(bp["boxes"], BAND_COLORS.values()):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
        ax.axhline(0, color="gray", lw=0.8, ls="--")
        if row == 0:
            caution = "  ⚠ funnel zone" if "500" in period else ""
            ax.set_title(f"{period}{caution}", fontsize=9)
        if col == 0:
            ax.set_ylabel(UNIT_LABEL[var], fontsize=9)
        ax.tick_params(axis="x", labelsize=7, rotation=35)

plt.suptitle("LMR period anomalies vs full-record mean (34-cell sample, by latitude band)",
             fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR / "11_period_anomalies.png", dpi=120)
plt.show()
print("Saved 11_period_anomalies.png")

Saved 11_period_anomalies.png


In [6]:
# Cell 6 — Volcanic event selection
# Restrict to reliable LMR window (700–1900 CE) and VSSI >= 20 Tg.
# Also require >=5yr before and >=5yr after event year within the LMR record (0–1998 CE).

reliable = evolv[(evolv.year_ad >= 700) & (evolv.year_ad <= 1900) & (evolv.vssi_tg >= 20)]
lag_events = reliable[
    (reliable.year_ad >= 705) & (reliable.year_ad <= 1993)
].sort_values("vssi_tg", ascending=False).head(8).reset_index(drop=True)

print(f"Events VSSI>=20 Tg in 700–1900 CE: {len(reliable)}")
print(f"Top 8 selected for lag-response analysis:")
print()
print(lag_events[["year_ad", "vssi_tg", "lat", "asymmetry", "location"]].to_string(index=False))

Events VSSI>=20 Tg in 700–1900 CE: 5
Top 8 selected for lag-response analysis:

 year_ad  vssi_tg   lat  asymmetry            location
    1257    59.42  -8.4      0.588 Samalas (Indonesia)
    1458    32.98 -16.8      0.357     Kuwae (Vanuatu)
    1815    28.08  -8.0      0.456 Tambora (Indonesia)
    1230    23.78   0.0      0.676                 NaN
    1783    20.81  64.4      1.000      Laki (Iceland)


In [7]:
# Cell 7 — Volcanic lag-response at individual cells
# Baseline = mean(ts, eruption_year - 5 to eruption_year - 1).
# Response = ts[eruption_year + lag] - baseline.
# Pooled across all 34 cells and all 8 events.

LAG_YEARS = [0, 1, 2, 3, 5]

lag_rows = []
for _, evt in lag_events.iterrows():
    ey = int(evt.year_ad)
    for i, cell in sample_df.iterrows():
        ts = series["air"][i]
        base_mask = (years >= ey - 5) & (years < ey)
        baseline = np.nanmean(ts[base_mask])
        for lag in LAG_YEARS:
            yr_idx = np.where(years == ey + lag)[0]
            if len(yr_idx) == 0:
                continue
            resp = float(ts[yr_idx[0]] - baseline)
            lag_rows.append({
                "event_year":    ey,
                "location":      str(evt.location)[:40],
                "vssi_tg":       float(evt.vssi_tg),
                "lag":           lag,
                "band":          cell["band"],
                "lat":           cell["lat"],
                "air_response_K": resp,
            })

lag_df = pd.DataFrame(lag_rows)

print("=== Median temperature response (K), pooled across all cells and events ===")
print(f"{'Lag (yr)':<12} {'Median response (K)':>20}  {'% cells cooler':>15}")
print("-" * 52)
for lag in LAG_YEARS:
    sub = lag_df[lag_df.lag == lag]
    med      = sub["air_response_K"].median()
    pct_cool = 100 * (sub["air_response_K"] < 0).mean()
    print(f"  {lag:<10} {med:>20.4f}  {pct_cool:>14.1f}%")

=== Median temperature response (K), pooled across all cells and events ===
Lag (yr)      Median response (K)   % cells cooler
----------------------------------------------------
  0                       -0.0509            64.1%
  1                        0.0381            41.2%
  2                       -0.0111            55.9%
  3                       -0.0261            59.4%
  5                       -0.0190            54.7%


In [8]:
# Cell 8 — Per-event temperature response plots (individual cells)
n_events = len(lag_events)
ncols = 4
nrows = (n_events + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows), sharey=True)
axes_flat = axes.ravel()

for idx, (_, evt) in enumerate(lag_events.iterrows()):
    ax = axes_flat[idx]
    ey = int(evt.year_ad)
    sub = lag_df[lag_df.event_year == ey]
    data_by_lag = [sub[sub.lag == lag]["air_response_K"].values for lag in LAG_YEARS]
    bp = ax.boxplot(data_by_lag, positions=list(range(len(LAG_YEARS))),
                    widths=0.6, patch_artist=True,
                    medianprops=dict(color="black", lw=1.5))
    for patch in bp["boxes"]:
        patch.set_facecolor("steelblue")
        patch.set_alpha(0.5)
    ax.axhline(0, color="gray", lw=0.8, ls="--")
    ax.set_xticks(range(len(LAG_YEARS)))
    ax.set_xticklabels([f"+{l}yr" for l in LAG_YEARS], fontsize=8)
    ax.set_title(f"{ey} CE  {evt.vssi_tg:.0f} Tg\n{str(evt.location)[:28]}", fontsize=8)
    if idx % ncols == 0:
        ax.set_ylabel("Temp response (K)", fontsize=8)

# Hide any unused subplots
for ax in axes_flat[n_events:]:
    ax.set_visible(False)

plt.suptitle(
    "LMR temperature response to major volcanic events (34-cell sample)\n"
    "Baseline = 5yr pre-eruption mean; boxes span IQR across cells",
    fontsize=10)
plt.tight_layout()
plt.savefig(OUT_DIR / "11_volcanic_response_cells.png", dpi=120)
plt.show()
print("Saved 11_volcanic_response_cells.png")

Saved 11_volcanic_response_cells.png


In [9]:
# Cell 9 — nhmt / gmt hemisphere-mean volcanic composite
# nhmt shape: (time=2001, MCrun=20, N=100).  Grand mean = nanmean over axes (1, 2).
# Composite: for each event, extract years -5 to +10 relative to eruption, subtract pre-eruption baseline.

print("Loading nhmt and gmt...", end="", flush=True)
ds_nh = nc.Dataset(NHMT_FILE)
nhmt_raw = np.ma.filled(ds_nh.variables["nhmt"][:], np.nan).astype(np.float32)
ds_nh.close()
ds_gmt = nc.Dataset(GMT_FILE)
gmt_raw = np.ma.filled(ds_gmt.variables["gmt"][:], np.nan).astype(np.float32)
ds_gmt.close()
print(" done")

# Grand mean over MCrun and ensemble-particle dimensions
nhmt_mean = np.nanmean(nhmt_raw, axis=(1, 2))   # shape (2001,)
gmt_mean  = np.nanmean(gmt_raw,  axis=(1, 2))
print(f"nhmt grand mean shape: {nhmt_mean.shape}  range [{nhmt_mean.min():.3f}, {nhmt_mean.max():.3f}] K")
print(f"gmt  grand mean shape: {gmt_mean.shape}   range [{gmt_mean.min():.3f}, {gmt_mean.max():.3f}] K")

# Build composite (lag -5 to +10)
LAG_RANGE = list(range(-5, 11))
nh_by_lag  = {lag: [] for lag in LAG_RANGE}
gmt_by_lag = {lag: [] for lag in LAG_RANGE}

for _, evt in lag_events.iterrows():
    ey = int(evt.year_ad)
    base_mask = (years >= ey - 5) & (years < ey)
    nh_base   = np.nanmean(nhmt_mean[base_mask])
    gmt_base  = np.nanmean(gmt_mean[base_mask])
    for lag in LAG_RANGE:
        yr_idx = np.where(years == ey + lag)[0]
        if len(yr_idx) == 0:
            continue
        nh_by_lag[lag].append(nhmt_mean[yr_idx[0]] - nh_base)
        gmt_by_lag[lag].append(gmt_mean[yr_idx[0]] - gmt_base)

valid_lags = [l for l in LAG_RANGE if len(nh_by_lag[l]) > 0]
nh_med     = [np.median(nh_by_lag[l])  for l in valid_lags]
gmt_med    = [np.median(gmt_by_lag[l]) for l in valid_lags]
nh_lo      = [np.percentile(nh_by_lag[l], 25) for l in valid_lags]
nh_hi      = [np.percentile(nh_by_lag[l], 75) for l in valid_lags]

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(valid_lags, nh_lo, nh_hi, alpha=0.25, color="steelblue", label="nhmt IQR")
ax.plot(valid_lags, nh_med,  color="steelblue",  lw=2.5, label="nhmt median")
ax.plot(valid_lags, gmt_med, color="firebrick",   lw=2.0, ls="--", label="gmt median")
ax.axvline(0, color="black", lw=1.2, ls=":", label="eruption year")
ax.axhline(0, color="gray",  lw=0.8, ls="--")
ax.set_xlabel("Years relative to eruption")
ax.set_ylabel("Temperature anomaly vs pre-eruption baseline (K)")
ax.set_title(
    f"LMR composite volcanic temperature response — {len(lag_events)} events ≥20 Tg, 700–1900 CE"
)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR / "11_volcanic_response_nhmt.png", dpi=120)
plt.show()

print("Composite NH temperature response (median K, relative to 5yr pre-eruption baseline):")
for lag, med in zip(valid_lags, nh_med):
    marker = " <-- eruption" if lag == 0 else ""
    print(f"  lag {lag:+3d}: {med:+.4f} K{marker}")
print("Saved 11_volcanic_response_nhmt.png")

Composite NH temperature response (median K, relative to 5yr pre-eruption baseline):
  lag  -5: +0.0469 K
  lag  -4: -0.0188 K
  lag  -3: +0.0272 K
  lag  -2: -0.0580 K
  lag  -1: +0.0265 K
  lag  +0: -0.0131 K <-- eruption
  lag  +1: +0.0980 K
  lag  +2: -0.0404 K
  lag  +3: -0.0462 K
  lag  +4: -0.0096 K
  lag  +5: -0.0337 K
  lag  +6: +0.0113 K
  lag  +7: +0.1786 K
  lag  +8: +0.0108 K
  lag  +9: +0.1225 K
  lag +10: +0.1177 K
Saved 11_volcanic_response_nhmt.png


In [10]:
# Cell 10 — Baseline convention comparison
# For MCA and LIA anomalies, compare three reference windows:
#   A: full record 0–1998 CE
#   B: reliable pre-industrial 1000–1850 CE  (candidate recommendation)
#   C: surrounding 200yr window centred on period midpoint (local reference)
# Goal: document how much the anomaly magnitude changes with reference choice.

QUERY_PERIODS = {
    "MCA (950–1250 CE)":  (950, 1250),
    "LIA (1300–1850 CE)": (1300, 1850),
}

rows = []
for var in ["air", "pdsi"]:
    for period_name, (p_lo, p_hi) in QUERY_PERIODS.items():
        p_mid = (p_lo + p_hi) // 2
        surr_lo = max(0,    p_mid - 100)
        surr_hi = min(1998, p_mid + 100)
        for i, cell in sample_df.iterrows():
            ts = series[var][i]
            p_mean = np.nanmean(ts[(years >= p_lo) & (years <= p_hi)])
            refs = {
                "Full record (0–1998)": np.nanmean(ts[(years >= 0)    & (years <= 1998)]),
                "Reliable pre-ind (1000–1850)": np.nanmean(ts[(years >= 1000) & (years <= 1850)]),
                f"Surrounding 200yr ({surr_lo}–{surr_hi})": np.nanmean(ts[(years >= surr_lo) & (years <= surr_hi)]),
            }
            for ref_name, r_mean in refs.items():
                rows.append({
                    "variable":  var,
                    "period":    period_name,
                    "reference": ref_name,
                    "anomaly":   p_mean - r_mean,
                    "lat":       cell["lat"],
                    "band":      cell["band"],
                })

bconv_df = pd.DataFrame(rows)

print("=== Anomaly by baseline convention (median across 34 cells) ===")
for period in QUERY_PERIODS:
    print(f"\n  {period}")
    for var in ["air", "pdsi"]:
        print(f"    {var}:")
        sub = bconv_df[(bconv_df.period == period) & (bconv_df.variable == var)]
        for ref_name, grp in sub.groupby("reference", sort=False):
            med = grp["anomaly"].median()
            iqr = grp["anomaly"].quantile(0.75) - grp["anomaly"].quantile(0.25)
            print(f"      {ref_name:<42} median {med:+.4f}  IQR {iqr:.4f}")

=== Anomaly by baseline convention (median across 34 cells) ===

  MCA (950–1250 CE)
    air:
      Full record (0–1998)                       median -0.0408  IQR 0.0414
      Reliable pre-ind (1000–1850)               median +0.0110  IQR 0.0540
      Surrounding 200yr (1000–1200)              median +0.0008  IQR 0.0142
    pdsi:
      Full record (0–1998)                       median +0.0266  IQR 0.0892
      Reliable pre-ind (1000–1850)               median +0.0094  IQR 0.1352
      Surrounding 200yr (1000–1200)              median +0.0000  IQR 0.0208

  LIA (1300–1850 CE)
    air:
      Full record (0–1998)                       median -0.0410  IQR 0.0415
      Reliable pre-ind (1000–1850)               median -0.0034  IQR 0.0168
      Surrounding 200yr (1475–1675)              median -0.0136  IQR 0.0190
    pdsi:
      Full record (0–1998)                       median -0.0087  IQR 0.0631
      Reliable pre-ind (1000–1850)               median -0.0089  IQR 0.0538
      Surrounding 2

In [ ]:
# Cell 11 — Save outputs
anom_df.to_csv(OUT_DIR   / "11_period_anomalies.csv",         index=False)
lag_df.to_csv(OUT_DIR    / "11_volcanic_response_cells.csv",  index=False)
bconv_df.to_csv(OUT_DIR  / "11_baseline_convention.csv",      index=False)

print("Task 11 outputs saved:")
expected = [
    "11_period_anomalies.csv",
    "11_volcanic_response_cells.csv",
    "11_baseline_convention.csv",
    "11_period_anomalies.png",
    "11_volcanic_response_cells.png",
    "11_volcanic_response_nhmt.png",
]
for fname in expected:
    p = OUT_DIR / fname
    size = p.stat().st_size if p.exists() else 0
    status = f"{size/1024:.0f} KB" if size else "NOT WRITTEN"
    print(f"  {fname:<45} {status}")

In [11]:
# Cell 12 — Location-specific validation: Kaifeng LMR cell across the Song dynasty
# Tests whether LMR shows a detectable cooling signal at a specific NH location after
# Samalas (1257 CE, 59 Tg) — the largest eruption in the LMR window.
# If yes: confirms the instrument can resolve the event at the regional scale relevant
#         to actual API queries, even where the global composite cannot.
# Second site: Central Europe (~48N, 10E) — NH coherence check across longitudes.
#
# Historical context:
#   Northern Song: 960–1127 CE (Jin conquest of Kaifeng, 1127)
#   Southern Song: 1127–1279 CE (Mongol conquest complete, 1279)
#   Samalas (1257): 22 years before the dynasty ends; 1258 documented in Chinese
#   sources as anomalous cold/crop failure; Mongol campaigns intensifying in this decade.

import pandas as pd

SITES = {
    "Kaifeng (~35°N, 114°E)":       (34.8,  114.3),
    "Central Europe (~48°N, 10°E)": (48.0,   10.0),
}
SITE_COLORS = {
    "Kaifeng (~35°N, 114°E)":       "steelblue",
    "Central Europe (~48°N, 10°E)": "darkorange",
}

def find_lmr_cell(target_lat, target_lon):
    lon_360 = target_lon % 360
    li = int(np.argmin(np.abs(lats - target_lat)))
    lj = int(np.argmin(np.abs(lons - lon_360)))
    return li, lj, float(lats[li]), float(lons[lj])

site_series = {}
for name, (tlat, tlon) in SITES.items():
    li, lj, slat, slon = find_lmr_cell(tlat, tlon)
    ts = extract_cell_mean(MEAN_FILES["air"], "air", li, lj)
    site_series[name] = {"ts": ts, "slat": slat, "slon": slon}
    print(f"{name}  →  snapped to ({slat:.0f}°N, {slon:.0f}°E)  [lat_idx={li}, lon_idx={lj}]")

# Reliable pre-industrial baseline (1000–1850 CE) — recommended convention from Cell 10
baseline_mask = (years >= 1000) & (years <= 1850)
for name in site_series:
    ts = site_series[name]["ts"]
    site_series[name]["baseline"] = np.nanmean(ts[baseline_mask])
    site_series[name]["anom"] = ts - site_series[name]["baseline"]

# eVolv2k events in Song window, VSSI >= 5 Tg
song_evts = evolv[(evolv.year_ad >= 960) & (evolv.year_ad <= 1280) &
                  (evolv.vssi_tg >= 5)].sort_values("year_ad")
print(f"\neVolv2k events >=5 Tg in Song dynasty (960–1280 CE): {len(song_evts)}")
print(song_evts[["year_ad", "vssi_tg", "location"]].to_string(index=False))

# ── Figure 1: Song dynasty overview (960–1280 CE) ──────────────────────────────
song_mask = (years >= 960) & (years <= 1280)
yr_song   = years[song_mask]

fig, ax = plt.subplots(figsize=(15, 5))
for name, color in SITE_COLORS.items():
    raw  = site_series[name]["anom"][song_mask]
    smth = pd.Series(raw).rolling(7, center=True, min_periods=4).mean().values
    ax.plot(yr_song, raw,  color=color, lw=0.6, alpha=0.35)
    ax.plot(yr_song, smth, color=color, lw=2.0, label=name)

ax.axhline(0, color="gray", lw=0.8, ls="--")

# Eruption markers: Samalas in red, others grey; label those >= 10 Tg
for _, evt in song_evts.iterrows():
    ey, vssi = int(evt.year_ad), float(evt.vssi_tg)
    is_samalas = vssi > 50
    col = "firebrick" if is_samalas else "#999999"
    lw  = 3.0 if is_samalas else 0.8
    ax.axvline(ey, color=col, lw=lw, ls="-", alpha=0.85 if is_samalas else 0.5, zorder=2)
    if vssi >= 10 or is_samalas:
        ax.text(ey + 1.5, -0.38, f"{int(vssi)}Tg", fontsize=7,
                color=col, rotation=90, va="bottom")

# Dynasty boundary and labels
ax.axvline(1127, color="black", lw=1.2, ls=":", alpha=0.6)
ax.text(1129, 0.30, "Jin conquest\n1127", fontsize=8, color="black", va="top")
ax.text(962,  0.30, "Northern Song", fontsize=9, color="black", style="italic")
ax.text(1138, 0.30, "Southern Song", fontsize=9, color="black", style="italic")
ax.text(1259, 0.27, "Samalas\n1257 · 59 Tg", fontsize=9,
        color="firebrick", ha="left", va="top", fontweight="bold")

ax.set_xlim(960, 1280)
ax.set_ylim(-0.5, 0.4)
ax.set_xlabel("Year CE")
ax.set_ylabel("Air temperature anomaly (K)\nvs 1000–1850 baseline")
ax.set_title("LMR air temperature — Song dynasty (960–1280 CE)\n"
             "7yr rolling mean · eruptions ≥5 Tg marked · Samalas (1257, 59 Tg) in red")
ax.legend(loc="lower left", fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR / "11_kaifeng_song_overview.png", dpi=120)
plt.show()

# ── Figure 2: Samalas close-up (1247–1272) ─────────────────────────────────────
samalas_yr = 1257
pre_mask   = (years >= samalas_yr - 5) & (years < samalas_yr)
window     = (years >= samalas_yr - 10) & (years <= samalas_yr + 15)
yr_win     = years[window]

fig, ax = plt.subplots(figsize=(12, 5))
for name, color in SITE_COLORS.items():
    ts   = site_series[name]["ts"]
    pre  = np.nanmean(ts[pre_mask])
    resp = ts[window] - pre
    ax.plot(yr_win, resp, color=color, lw=2.0, marker="o", ms=4, label=name)
    ax.axhline(0, color=color, lw=0.7, ls="--", alpha=0.4)

ax.axvline(samalas_yr, color="firebrick", lw=2.0, ls=":",
           label=f"Samalas {samalas_yr} (59 Tg)")
ax.axvspan(samalas_yr, samalas_yr + 3, alpha=0.07, color="firebrick",
           label="Expected cooling window (+0 to +3yr)")
ax.set_xlabel("Year CE")
ax.set_ylabel("Temperature response (K)\nvs 5yr pre-eruption mean")
ax.set_title(f"LMR temperature response — Samalas {samalas_yr} CE (59 Tg)\n"
             f"Baseline = {samalas_yr - 5}–{samalas_yr - 1} CE mean · "
             "two NH locations")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "11_kaifeng_samalas_closeup.png", dpi=120)
plt.show()

# Values table — print after plt.show()
kf_ts  = site_series["Kaifeng (~35°N, 114°E)"]["ts"]
kf_pre = np.nanmean(kf_ts[pre_mask])
eu_ts  = site_series["Central Europe (~48°N, 10°E)"]["ts"]
eu_pre = np.nanmean(eu_ts[pre_mask])
print(f"Kaifeng and Central Europe: temperature response vs {samalas_yr-5}–{samalas_yr-1} baseline")
print(f"{'Year':<8} {'Kaifeng (K)':>14}  {'C. Europe (K)':>15}")
print("-" * 42)
for yr in range(samalas_yr - 3, samalas_yr + 8):
    idx = np.where(years == yr)[0]
    if len(idx) == 0:
        continue
    kf_r = kf_ts[idx[0]] - kf_pre
    eu_r = eu_ts[idx[0]] - eu_pre
    marker = "  ← eruption" if yr == samalas_yr else ""
    print(f"{yr:<8} {kf_r:>14.4f}  {eu_r:>15.4f}{marker}")
print("\nSaved 11_kaifeng_song_overview.png, 11_kaifeng_samalas_closeup.png")

Kaifeng and Central Europe: temperature response vs 1252–1256 baseline
Year        Kaifeng (K)    C. Europe (K)
------------------------------------------
1254             0.2702           0.2761
1255            -0.1042          -0.1515
1256             0.0545           0.0686
1257            -0.1321          -0.4321  ← eruption
1258             0.0226          -0.1458
1259             0.1250          -0.0471
1260            -0.1246          -0.0681
1261            -0.0834          -0.0560
1262            -0.2410          -0.0948
1263            -0.1577          -0.2808
1264            -0.0896          -0.2728

Saved 11_kaifeng_song_overview.png, 11_kaifeng_samalas_closeup.png
